## Data Preprocessing

In [12]:
# Importing the basic libraries for data manipulation and preprocessing

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [13]:
# Load the dataset
df = pd.read_csv('../../data/raw/ai4i2020.csv')

# Step 1: Check for missing values
print("Missing values per column:")
print(df.isna().sum())
print("\n" + "="*60 + "\n")

Missing values per column:
UID                    0
Product ID             0
Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
Machine failure        0
TWF                    0
HDF                    0
PWF                    0
OSF                    0
RNF                    0
dtype: int64




Since there is no missing values, we are good to move to next step

In [14]:
# Step 2: Extract product quality variant from Product ID
# Product ID format: e.g., "L50001" -> quality = "L", serial = "50001"
df["product_quality"] = df["Product ID"].str[0]  # Extract first letter
print("Unique product quality variants:")
print(df["product_quality"].unique())
print("Product quality distribution:")
print(df["product_quality"].value_counts())


Unique product quality variants:
<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str
Product quality distribution:
product_quality
L    6000
M    2997
H    1003
Name: count, dtype: int64


Why: Quality variant matters because:

Different quality products accumulate tool wear differently (L adds 2 min, M adds 3 min, H adds 5 min).
This influences failure modes like overstrain.
The model needs to know product quality to predict correctly.

In [15]:
# Step 3: Check target variable balance BEFORE preprocessing
print("Target variable balance (before split):")
print(df["Machine failure"].value_counts())
print(f"Failure rate: {df['Machine failure'].mean():.2%}")

Target variable balance (before split):
Machine failure
0    9661
1     339
Name: count, dtype: int64
Failure rate: 3.39%


In [16]:
# Since, there is a significant imbalance in the target variable, we will need to address this issue during the data preprocessing steps. 
# Only 3.39% of the samples represent machine failures, which could lead to biased model performance if not handled properly. 
# A naive model that predicts "no failure" for everything would be 96% accurate but useless. 
# This is why recall is more important than accuracy: missing a failure is costly and unsafe.
# We might need to use techniques like: class_weight='balanced' in the model, undersampling the majority class, oversampling the minority class.

In [34]:
# Step 4: Preparing features and target

input_features = [
    "Type",
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]

label_columns = ["Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"]

X = df[input_features].copy()
y = df["Machine failure"].copy()

failure_modes = df[["TWF", "HDF", "PWF", "OSF", "RNF"]].copy()

print("Input feature columns:")
print(X.columns.tolist())

print("\nTarget column:")
print("Machine failure")

print("\nFailure mode columns kept only for analysis/evaluation:")
print(failure_modes.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Input feature columns:
['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear']

Target column:
Machine failure

Failure mode columns kept only for analysis/evaluation:
['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

X shape: (10000, 6)
y shape: (10000,)


There are 12 input features:

Sensor readings: Air temperature, Process temperature, Rotational speed, Torque, Tool wear

Failure mode indicators: TWF, HDF, PWF, OSF, RNF (binary flags showing if each mode occurred)


# Step 6: Stratified train/test split

Key detail: stratified split

stratify=y ensures the failure proportion stays ~3.39% in both train and test.

Without stratification, the test set might have 1% failures (not representative) or 10% (biased).

This is critical for imbalanced data to ensure fair evaluation.

In [39]:
# Step 5: Train/test split before encoding and scaling
# Stratification keeps failure/no-failure proportions consistent in both sets

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set size: {X_train_raw.shape[0]} samples")
print(f"Test set size: {X_test_raw.shape[0]} samples")
print(f"\nTraining set failure rate: {y_train.mean():.2%}")
print(f"Test set failure rate: {y_test.mean():.2%}")

Training set size: 8000 samples
Test set size: 2000 samples

Training set failure rate: 3.39%
Test set failure rate: 3.40%


# Encode Categorical Feature

Categorical: Type (product type), product_quality

One-hot encoding converts product_quality (L/M/H) into numeric columns:

product_quality_L: 1 if L, else 0
    
product_quality_M: 1 if M, else 0

(H is implicit: 0 for both means H)

In [44]:
# Step 6: Encode categorical feature Type

X_train_encoded = pd.get_dummies(X_train_raw, columns=["Type"], drop_first=True, dtype=int)
X_test_encoded = pd.get_dummies(X_test_raw, columns=["Type"], drop_first=True, dtype=int)

# Make sure train and test have the same columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)

print("Encoded training columns:")
print(X_train_encoded.columns.tolist())

print("\nTrain shape:", X_train_encoded.shape)
print("Test shape:", X_test_encoded.shape)

Encoded training columns:
['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'Type_L', 'Type_M']

Train shape: (8000, 7)
Test shape: (2000, 7)


# Scale numeric sensor features

In [48]:
# Step 7: Scale numeric sensor features

numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]

# Initialize the StandardScaler to scale numeric features of both test and train sets to have mean~0 and std=1
scaler = StandardScaler()

X_train_processed = X_train_encoded.copy()
X_test_processed = X_test_encoded.copy()

X_train_processed[numeric_features] = scaler.fit_transform(
    X_train_encoded[numeric_features]
)

X_test_processed[numeric_features] = scaler.transform(
    X_test_encoded[numeric_features]
)

print("Processed training data:")
print(X_train_processed.head())

print("\nProcessed test data:")
print(X_test_processed.head())

print("\n Train data mean and std after scaling:")
print(X_train_processed[numeric_features].mean())
print("\n")
print(X_train_processed[numeric_features].std())

print("\n Test data mean and std after scaling:")
print(X_test_processed[numeric_features].mean())
print("\n")
print(X_test_processed[numeric_features].std())

Processed training data:
      Air temperature  Process temperature  Rotational speed    Torque  \
4058         0.998914             0.604282         -0.460607  0.718305   
1221        -1.505194            -1.153260         -0.775574  0.638456   
6895         0.498092             1.077466         -1.007654  0.558607   
9863        -0.553633            -0.139294         -0.709265  1.626586   
8711        -1.455112            -1.018064          1.070019 -1.128202   

      Tool wear  Type_L  Type_M  
4058  -0.843997       0       1  
1221   0.382263       0       1  
6895   0.460870       0       1  
9863  -0.372359       1       0  
8711  -0.906882       1       0  

Processed test data:
      Air temperature  Process temperature  Rotational speed    Torque  \
2997         0.247681            -0.139294         -1.073963  2.265378   
4871         1.850310             1.618248         -0.145641  0.009646   
3858         1.249324             0.942270          0.108543 -0.239882   
951     